# Last Mile Delivery Report Automation by Delivery Time

**Core ETL Components:**
1.  **Extract:** Uploads the raw CSV file and loads it into a Pandas DataFrame.
2. **Transform:** Clean data types, filter specific columns, sort logically, and apply conditional business formatting (B2B & COD orders).
3. **Load:** Distribute the processed data into delivery time-specific Excel files and grouped by 'hubs', format columns, and compress into a single ZIP archive for distribution.

## **HOW TO USE**
### 🐍 1. Python (Data Processing & ETL)

These scripts are designed for **Google Colab.**

1. **Open:** Upload the `.ipynb` file to [Google Colab](https://colab.research.google.com/).
2. **Input:** Download the raw `.csv` file sample provided in `/data`.
3. **Run:** Select `Runtime > Run All` from the top menu.
4. **Output:** 
    1. Last mile delivery reconciliation data: the cleaned and formatted `.xlsx` files will be generated and downloaded automatically.
    2. Fleets: the cleaned CSV file and data visualization.

# Setup, Imports, and Configuration

In [ ]:
import os
import shutil
from zipfile import ZipFile
from datetime import datetime
import pandas as pd
import numpy as np
import pytz
import openpyxl
from google.colab import files

# --- CONFIGURATION ---
LOCAL_TZ = pytz.timezone('Asia/Jakarta')
DATE_FORMAT = "%d-%m-%Y"
CURRENT_DATE_STR = datetime.now(LOCAL_TZ).strftime(DATE_FORMAT)

UPLOAD_DIR = '/content/'
EXPORT_DIR = '/content/exported_files'

# Desired columns for the final output
TARGET_COLUMNS = [
    'driver_name', 'driver_phone', 'district', 'customer_name', 'customer_email',
    'delivery_date', 'address', 'customer_phone', 'order_no',
    'customer_status', 'packaging_option', 'distance_in_km',
    'Zone_Name', 'Zone_Price', 'hubs', 'total_price', 'time_slot',
    'payment_method', 'total_weight_perorder', 'shipping_number (Box #)'
    ]


# Ensure export directory exists
os.makedirs(EXPORT_DIR, exist_ok=True)
print(f"Setup complete. Current processing date: {CURRENT_DATE_STR}")

# 1. Extract
Upload the raw CSV file into the Colab environment and load it into a Pandas DataFrame.

In [ ]:
def extract_data(upload_dir: str) -> pd.DataFrame:
    print("Please upload the daily delivery CSV file:")
    uploaded = files.upload()
    
    csv_files = [f for f in os.listdir(upload_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in the upload directory.")
        
    file_path = os.path.join(upload_dir, csv_files[0])
    df = pd.read_csv(file_path)
    print(f"Successfully extracted {len(df)} rows from {csv_files[0]}")
    return df

# Execute Extract
raw_df = extract_data(UPLOAD_DIR)

# 2. Transform
Clean data types, filter the necessary columns, and sort the data for operational efficiency.

In [ ]:
def clean_and_transform(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Applies type casting, column filtering, and sorting."""
    # Always operate on a copy to preserve raw data
    transformed_df = df.copy()

    # 1. Transform columns
    # Transform values in 'total_price' column by filling nan values with 0 and convert data type into integer
    transformed_df.loc[:, 'total_price'] = (
        transformed_df['total_price']
        .fillna(0)
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype(float)
        .astype(int)
    )
    # Convert 'driver_name' to uppercase and remove 'MITRA-' or 'mitra-' prefix
    transformed_df.loc[:, 'driver_name'] = (
        transformed_df['driver_name']
        .str.upper()
        .str.replace('^(MITRA-|mitra-)', '', regex=True)
        )
    # Replace 'ALL' in 'packaging_option' with 'BIGBOX + BIGBOX'
    transformed_df.loc[:, 'packaging_option'] = (
        transformed_df['packaging_option']
        .str.replace('^(ALL)', 'BIGBOX + BIGBOX', regex=True)
        )
    # Round 'distance_in_km' to 2 decimal places
    transformed_df.loc[:, 'distance_in_km'] = (
        transformed_df['distance_in_km'].round(2)
        )
    # Replace specific time slot variations with standardized values
    transformed_df.loc[:, 'time_slot'] = (
        transformed_df['time_slot']
        .str.replace('^(slot-12bb)', 'slot-1bb', regex=True)
        .str.replace('^(slot-b2b-2)', 'slot-0bb', regex=True)
        )
    
    # 3. Filter Columns
    valid_columns = [col for col in columns if col in transformed_df.columns]
    transformed_df = transformed_df[valid_columns]

    # 4. Sort Data logically for logistics
    if all(col in transformed_df.columns for col in ['hubs', 'driver_name', 'customer_name']):
        transformed_df = transformed_df.sort_values(
            by=['hubs', 'driver_name', 'customer_name'], 
            ascending=[True, True, True]
        )

    return transformed_df

# Execute Transform
clean_df = clean_and_transform(raw_df, TARGET_COLUMNS)
display(clean_df.head(3))

# 3. Formatting & Load Phase (Time Slot Distribution)
Apply business-logic styling (B2B & COD) and split the master dataframe into vendor-specific Excel files.

In [ ]:
# Define a function to apply conditional highlighting and borders to rows
def highlight_b2b_and_payment(row: pd.Series) -> list[str]:
  """
  Applies conditional highlighting and borders to a DataFrame row.

  - Rows with 'time_slot' as 'slot-0bb' or 'slot-1bb' will have a light blue background.
  - Rows with 'payment_method' as 'Cash on Delivery' will have green text.
  - All cells will have a light gray border.

  Args:
    row (pd.Series): A row from a DataFrame.

  Returns:
    list: A list of style strings for each cell in the row.
  """
  # Get the values from the 'time_slot' and 'payment_method' columns for the current row
  time_slot_value = row.loc['time_slot']
  payment_method_value = row.loc['payment_method']

  # Initialize a list to store styles for each cell in the row
  styles = []
  # Iterate through each item (cell value) in the current row
  for item in row:
    # Define default background and text colors, and border style
    background_color = '#FFFFFF'  # Default background color (white)
    text_color = '#000000'  # Default text color (black)
    border_style = '0.5px solid #D3D3D3' # Define border style

    # Apply light blue background if the time slot is 'slot-0bb' or 'slot-1bb'
    if time_slot_value in ('slot-0bb', 'slot-1bb'):
      background_color = '#7393B3'  # Light blue color

    # Apply green text color if the payment method is 'Cash on Delivery'
    # Using 'if' here allows this style to be applied in addition to the background color if both conditions are met
    if payment_method_value == 'Cash on Delivery':
      text_color = 'green'  # Green text color

    # Append the combined style string for the current cell to the styles list
    styles.append(f'background-color: {background_color}; color: {text_color}; border: {border_style}')

  # Return the list of style strings for the row
  return styles

# 4. Post-Processing & Archiving
Zip files for download.

In [ ]:
# Define a function to adjust column width and center align text in an Excel file
def adjust_excel_columns(filename: str):
    """
    Loads an Excel workbook, center aligns text in all cells of the header row,
    applies left alignment to columns A and C from the second row onwards,
    auto-adjusts column widths for specific columns, and saves the modified workbook.

    Args:
        filename (str): The path to the Excel file.
    """
    try:
        # Load the workbook
        workbook = openpyxl.load_workbook(filename)
        delivery_date_col_idx = None  # Safe default fallback
        
        # Iterate through all sheets in the workbook
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]

            # Iterate through rows and columns to set alignment and date format
            for row_index, row in enumerate(sheet.iter_rows()):
                for cell in row:
                    if row_index == 0:  # Apply center alignment to the header row
                        cell.alignment = openpyxl.styles.Alignment(horizontal='center', vertical='center')
                    elif cell.column_letter in ('A', 'C'):  # Apply left alignment to columns A and C from the second row
                        cell.alignment = openpyxl.styles.Alignment(horizontal='left', vertical='center')

                    # Find the column index for 'delivery_date' if the header row is processed
                    if row_index == 0 and cell.value == 'delivery_date':
                      delivery_date_col_idx = cell.column - 1 # Get the 0-based index of the delivery_date column


            # Apply date format to the 'delivery_date' column from the second row onwards
            for row_index in range(1, sheet.max_row): # Start from the second row (index 1)
              cell = sheet.cell(row=row_index + 1, column=delivery_date_col_idx + 1) # Get the cell in the delivery_date column
              cell.number_format = 'YYYY-MM-DD' # Apply the date format

            # Auto-adjust column width for specific columns in the current sheet
            for column in sheet.columns:
                column_letter = openpyxl.utils.get_column_letter(column[0].column)
                # Apply width adjustment only to columns A and C
                if column_letter in ('A', 'C'):
                    max_length = 0
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    adjusted_width = (max_length + 1.5)
                    sheet.column_dimensions[column_letter].width = adjusted_width


        # Save the modified workbook
        workbook.save(filename)

        print(f"Text in '{filename}' has been aligned and columns adjusted for all sheets.")

    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
    except Exception as e:
        print(f"An error occurred while processing '{filename}': {e}")

In [ ]:
def export_delivery_list_to_excel(df: pd.DataFrame, time_slot_pattern: str, slot_name: str, sheet_suffix: str):
    # Filter df to include only rows where 'time_slot' matches the given regex
    filter_by_time_slot = df[df['time_slot'].str.contains(time_slot_pattern, case=True)]
    
    if filter_by_time_slot.empty:
        print(f"No data for {slot_name} ({time_slot_pattern}). Skipping export.")
        return
    
    # Group the filtered DataFrame by the 'hubs' column
    groups = filter_by_time_slot.groupby('hubs')
    
    # Create an Excel writer object with a dynamic filename based on the current date and slot
    filename = f"List Delivery {CURRENT_DATE_STR} {slot_name}.xlsx"
    writer = pd.ExcelWriter(filename)

    # Iterate through each group (each unique 'hubs' value)
    for hubs, group_df in groups:
        # Define the sheet name for the current group
        sheet_name = f"{hubs}{sheet_suffix}"
        # Apply conditional formatting and borders to the current group's DataFrame
        formatted_df = group_df.style \
        .apply(highlight_b2b_and_payment, axis=1) \
        .set_table_styles([
          {'selector': '', 'props': [('border', '0.5px solid #D3D3D3')]}
      ])
        # Write the formatted group DataFrame to a sheet in the Excel file
        # index=False prevents writing the DataFrame index to the Excel file
        formatted_df.to_excel(writer, sheet_name=sheet_name, index=False)

    # Save the Excel file by closing the writer object
    writer.close()

    # Call the function to adjust columns after creating the Excel file
    adjust_excel_columns(filename)
    
    # Print a success message indicating the filename of the exported file
    print(f"{filename} is exported successfully")
    
    # Create the destination folder if it doesn't exist.
    os.makedirs(EXPORT_DIR, exist_ok=True)
    # Move the exported file to the destination folder.
    shutil.move(filename, EXPORT_DIR)
    
    filepath = os.path.join(EXPORT_DIR, filename)
    
    # Print a message indicating where the file was moved.
    print(f"File moved to: {filepath}")

In [ ]:
# Define a function to zip all files within a specified folder.
def zip_exported_files(folder_path, zip_filename):
  """Zips all files within a folder.

  Args:
      folder_path (str): Path to the folder containing the files.
      zip_filename (str): Name of the output zip file.
  """
  # Create a zip file in write mode.
  with ZipFile(zip_filename, 'w') as zipf:
    # Walk through the directory tree starting from the folder_path.
    for root, _, files in os.walk(folder_path):
      # Iterate through each file in the current directory.
      for file in files:
        # Get the full path of the file.
        file_path = os.path.join(root, file)
        # Write the file to the zip archive, maintaining the relative path within the zip.
        zipf.write(file_path, os.path.relpath(file_path, folder_path))

In [ ]:
# Call the function for SLOT-0
export_delivery_list_to_excel(clean_df, 'slot-0|slot-0bb|slot-b2b-2', 'SLOT-0', '0')

# Call the function for SLOT-1
export_delivery_list_to_excel(clean_df, 'slot-1$|slot-1bb|slot-12bb', 'SLOT-1', '1')

# Call the function for SLOT-2
export_delivery_list_to_excel(clean_df, 'slot-2|slot-sameday03', 'SLOT-2', '2')

# Call the function for SLOT-13
export_delivery_list_to_excel(clean_df, 'slot-13|slot-sameday$', 'SLOT-13', '13')

In [ ]:
# Define the name for the output zip file.
zip_filename = f'exported_files_{CURRENT_DATE_STR}.zip' 

# Call the function to zip the exported files.
zip_exported_files(folder_path=EXPORT_DIR, zip_filename=zip_filename)
# Print a confirmation message.
print(f"Folder '{EXPORT_DIR}' zipped to '{zip_filename}'")

# Download the created zip file.
files.download(zip_filename)